# Experiment: ForteBank Dataset Audit

Objective: understand the supplied videos before choosing a model. Success means every video is matched to its label, durations and frame rates are known, and labeled events can be inspected as short visual windows.


In [ ]:
from __future__ import annotations

import os
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
sys.path.insert(0, str(PROJECT_ROOT / 'src'))
DATA_ROOT = Path(os.environ.get('FORTEBANK_DATA_ROOT', PROJECT_ROOT / 'data' / 'raw'))
ARTIFACTS = PROJECT_ROOT / 'artifacts' / 'dataset_audit'

from fortebank_audit import build_index, extract_frame, probe_video

PROJECT_ROOT, DATA_ROOT


## Plan

1. Validate that labels and videos match one-to-one.
2. Measure class balance, duration, FPS, and resolution.
3. Inspect five-frame windows around positive timestamps.
4. Write down visually observable event categories before training.


In [ ]:
records = build_index(DATA_ROOT)
index_df = pd.DataFrame(record.as_row() for record in records)
summary = {
    'videos': len(index_df),
    'positive': int(index_df['has_violation'].sum()),
    'negative': int((~index_df['has_violation']).sum()),
    'positive_with_timecode': int(index_df.loc[index_df['has_violation'], 'timecode_sec'].notna().sum()),
}
summary


In [ ]:
metadata_df = pd.DataFrame(probe_video(path) for path in index_df['path'])
audit_df = pd.concat([index_df.reset_index(drop=True), metadata_df.drop(columns='path')], axis=1)
audit_df[['video_id', 'has_violation', 'timecode_sec', 'duration_sec', 'fps', 'width', 'height']].head(10)


In [ ]:
audit_df.groupby('has_violation')['duration_sec'].agg(['count', 'min', 'median', 'max']).round(2)


## Inspect labeled event windows

Each row below shows frames at -4, -2, 0, +2, and +4 seconds relative to the supplied timestamp. Use these windows to define event categories from visible actions only.


In [ ]:
def show_event(record, offsets=(-4, -2, 0, 2, 4)):
    metadata = probe_video(record.path)
    center = record.timecode_sec or 0.0
    timestamps = [min(max(0.0, center + offset), metadata['duration_sec'] - 0.05) for offset in offsets]
    figure, axes = plt.subplots(1, len(timestamps), figsize=(18, 4))
    for axis, timestamp in zip(axes, timestamps):
        axis.imshow(extract_frame(record.path, timestamp))
        axis.set_title(f'{timestamp:.1f}s')
        axis.axis('off')
    figure.suptitle(f'{record.display_name} | label={record.raw_label} | event={center:.1f}s')
    figure.tight_layout()
    return figure

positive_records = [record for record in records if record.has_violation]
show_event(positive_records[0]);


In [ ]:
for record in positive_records[:6]:
    show_event(record)
    plt.show()


## Results and next decision

Record the first event taxonomy here after reviewing all positive windows:

| Event category | Visible evidence | Example video ids | Ambiguities |
|---|---|---|---|
| TODO | TODO | TODO | TODO |

Next: choose a local baseline only after the event taxonomy is stable enough to distinguish object presence, hand interaction, and temporal sequence errors.
